# Preview

- Use these questions before class to check your familiarity with the chapter glossary and core terms.
- Sign in from the account menu before you start answering the questions.
- Multiple submissions are allowed. The highest score is kept.


<style>
.ch02-preview-quiz { max-width: 900px; }
.ch02-preview-question { border: 1px solid #d8dee4; border-radius: 6px; padding: 1rem; margin: 1rem 0; background: #fff; }
.ch02-preview-question p { margin-top: 0; font-weight: 600; }
.ch02-preview-question label { display: block; margin: 0.35rem 0; cursor: pointer; }
.ch02-preview-question input { margin-right: 0.4rem; }
.ch02-preview-actions { display: flex; gap: 0.75rem; align-items: center; margin: 1rem 0; flex-wrap: wrap; }
.ch02-preview-actions button { border: 1px solid #176de8; border-radius: 6px; background: #176de8; color: white; padding: 0.55rem 0.9rem; cursor: pointer; font-weight: 600; }
.ch02-preview-actions button.secondary { background: white; color: #176de8; }

.ch02-preview-actions.is-locked { opacity: 0.55; }
.ch02-preview-actions button:disabled { cursor: not-allowed; opacity: 0.65; }
.ch02-preview-result { font-weight: 600; }
.ch02-preview-feedback { margin-top: 0.65rem; font-weight: 600; }
.ch02-preview-feedback.correct { color: #1a7f37; }
.ch02-preview-feedback.incorrect { color: #cf222e; }
</style>

<div class='ch02-preview-quiz' id='ch02-preview-quiz'>
  <div id='ch02-preview-questions'></div>
  <div class='ch02-preview-actions'>
    <button type='button' id='ch02-preview-submit' disabled>Submit</button>
    <button type='button' class='secondary' id='ch02-preview-reset'>Reset</button>
    <span class='ch02-preview-result' id='ch02-preview-result' aria-live='polite'></span>
  </div>
</div>

<script>
(function () {
  var quizId = 'ch02-preview';
  var questions = [
    { id: 'q1', prompt: 'Which term means the formal rules that define how Python statements must be written?', choices: { A: 'Syntax', B: 'Scope', C: 'Argument', D: 'Dictionary' } },
    { id: 'q2', prompt: 'Which term means changing a value from one type to another, such as converting a string to an integer?', choices: { A: 'Assignment statement', B: 'Type conversion', C: 'Boolean expression', D: 'Aliasing' } },
    { id: 'q3', prompt: 'Which term means the order Python follows when evaluating operations in an expression?', choices: { A: 'Dynamic typing', B: 'Method', C: 'Operator precedence', D: 'Stack diagram' } },
    { id: 'q4', prompt: 'Which term means code that runs different instructions depending on whether a condition is true or false?', choices: { A: 'Conditional statement', B: 'Sequence', C: 'Package', D: 'Function call' } },
    { id: 'q5', prompt: 'Which term means a special value that signals when a loop should stop?', choices: { A: 'Index', B: 'Sentinel value', C: 'Parameter', D: 'Docstring' } },
    { id: 'q6', prompt: 'Which term means a position number used to access an item in a sequence?', choices: { A: 'Key', B: 'Index', C: 'Accumulator', D: 'Keyword' } },
    { id: 'q7', prompt: 'Which term means the ability of an object, such as a list, to be changed after it is created?', choices: { A: 'Hashable', B: 'Mutability', C: 'Return value', D: 'Iteration' } },
    { id: 'q8', prompt: 'Which term means a function that belongs to an object, such as append() for a list?', choices: { A: 'Method', B: 'Expression', C: 'Nested conditional', D: 'Infinite loop' } },
    { id: 'q9', prompt: 'Which term means one entry in a dictionary, made from a key and the value associated with it?', choices: { A: 'Key-value pair', B: 'Function', C: 'Operator', D: 'Object' } },
    { id: 'q10', prompt: 'Which term means the part of a program where a variable name can be used?', choices: { A: 'List', B: 'Traceback', C: 'For loop', D: 'Scope' } }
  ];

  function text(value) {
    return document.createTextNode(value);
  }

  function getParams() {
    var params = new URLSearchParams(window.location.search);
    return {
      canvas_course_id: params.get('canvas_course_id') || params.get('course_id') || '',
      canvas_assignment_id: params.get('canvas_assignment_id') || params.get('assignment_id') || '',
      canvas_user_id: params.get('canvas_user_id') || params.get('user_id') || '',
      student_identifier: params.get('student_identifier') || params.get('student_id') || params.get('email') || ''
    };
  }

  function loadSessionIdentity(identity, actions, submit, result) {

    submit.disabled = true;

    actions.classList.add('is-locked');

    result.textContent = 'Sign in from the account menu to submit this assignment.';

    return fetch('/api/v1/session.php', { credentials: 'same-origin' })

      .then(function (response) { return response.ok ? response.json() : null; })

      .then(function (payload) {

        if (!payload || !payload.authenticated || !payload.identity) return identity;

        identity.canvas_user_id = payload.identity.canvas_user_id || identity.canvas_user_id || '';

        identity.student_identifier = payload.identity.student_identifier || identity.student_identifier || '';

        submit.disabled = false;

        actions.classList.remove('is-locked');

        result.textContent = payload.identity.display_name ? 'Signed in as ' + payload.identity.display_name + '.' : 'Signed in.';

        return identity;

      })

      .catch(function () { return identity; });

  }



  function renderQuestions(container) {
    questions.forEach(function (question, index) {
      var card = document.createElement('div');
      card.className = 'ch02-preview-question';
      card.dataset.question = question.id;

      var prompt = document.createElement('p');
      prompt.appendChild(text((index + 1) + '. ' + question.prompt));
      card.appendChild(prompt);

      Object.keys(question.choices).forEach(function (letter) {
        var label = document.createElement('label');
        var input = document.createElement('input');
        input.type = 'radio';
        input.name = question.id;
        input.value = letter;
        label.appendChild(input);
        label.appendChild(text(letter + '. ' + question.choices[letter]));
        card.appendChild(label);
      });

      var feedback = document.createElement('div');
      feedback.className = 'ch02-preview-feedback';
      feedback.setAttribute('aria-live', 'polite');
      card.appendChild(feedback);
      container.appendChild(card);
    });
  }

  function collectAnswers(quiz) {
    var answers = {};
    questions.forEach(function (question) {
      var selected = quiz.querySelector('input[name=' + question.id + ']:checked');
      answers[question.id] = selected ? selected.value : '';
    });
    return answers;
  }

  function applyFeedback(quiz, feedback) {
    questions.forEach(function (question) {
      var card = quiz.querySelector('[data-question=' + question.id + ']');
      var message = card.querySelector('.ch02-preview-feedback');
      var item = feedback[question.id];
      message.className = 'ch02-preview-feedback';
      if (!item || !item.submitted) {
        message.textContent = item && item.message ? item.message : 'No answer submitted.';
        message.classList.add('incorrect');
      } else if (item.correct) {
        message.textContent = item.message || 'Correct.';
        message.classList.add('correct');
      } else {
        message.textContent = item.message || 'Try again.';
        message.classList.add('incorrect');
      }
    });
  }

  function initPreviewQuiz() {
    var quiz = document.getElementById('ch02-preview-quiz');
    if (!quiz || quiz.dataset.ready === 'true') return;
    quiz.dataset.ready = 'true';

    var identity = getParams();
    var questionsContainer = document.getElementById('ch02-preview-questions');
    var result = document.getElementById('ch02-preview-result');
    var submit = document.getElementById('ch02-preview-submit');
    var actions = quiz.querySelector('.ch02-preview-actions');
    var reset = document.getElementById('ch02-preview-reset');

    renderQuestions(questionsContainer);
    loadSessionIdentity(identity, actions, submit, result);

    submit.addEventListener('click', function () {
      var answers = collectAnswers(quiz);
      submit.disabled = true;
      result.textContent = 'Submitting...';

      fetch('/api/v1/quiz-attempts.php', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        credentials: 'same-origin',
        body: JSON.stringify({
          quiz_id: quizId,
          answers: answers,
          student_identifier: identity.student_identifier || identity.canvas_user_id || '',
          canvas_course_id: identity.canvas_course_id,
          canvas_assignment_id: identity.canvas_assignment_id,
          canvas_user_id: identity.canvas_user_id
        })
      })
        .then(function (response) {
          return response.text().then(function (text) {
            var payload = null;
            try {
              payload = text ? JSON.parse(text) : null;
            } catch (error) {
              throw new Error('Server returned a non-JSON response (HTTP ' + response.status + '): ' + text.slice(0, 160));
            }
            if (!payload) {
              throw new Error('Server returned an empty response (HTTP ' + response.status + '). Check that PHP is enabled for /api/v1/quiz-attempts.php.');
            }
            if (!response.ok || !payload.ok) {
              throw new Error(payload.message || payload.error || 'Submission failed.');
            }
            return payload;
          });
        })
        .then(function (payload) {
          var best = (payload.best_score === null || typeof payload.best_score === 'undefined') ? payload.score : payload.best_score;
          var attempt = (payload.attempt_count === null || typeof payload.attempt_count === 'undefined') ? '' : ' Attempt ' + payload.attempt_count + '.';
          applyFeedback(quiz, payload.feedback || {});
          result.textContent = 'Submission saved.' + attempt + ' Score: ' + payload.score + ' / ' + payload.max_score + '. Best score: ' + best + ' / ' + payload.max_score + '. Canvas sync: ' + payload.canvas_sync_status + '.';
        })
        .catch(function (error) {
          result.textContent = 'Could not save this attempt: ' + error.message;
        })
        .finally(function () {
          submit.disabled = false;
        });
    });

    reset.addEventListener('click', function () {
      quiz.querySelectorAll('input[type=radio]').forEach(function (input) {
        input.checked = false;
      });
      quiz.querySelectorAll('.ch02-preview-feedback').forEach(function (feedback) {
        feedback.className = 'ch02-preview-feedback';
        feedback.textContent = '';
      });
      result.textContent = '';
    });
  }

  if (document.readyState === 'loading') {
    document.addEventListener('DOMContentLoaded', initPreviewQuiz);
  } else {
    initPreviewQuiz();
  }
})();
</script>